# 🔧 Decision Tree Model - RETRAINED & FINE-TUNED
## Fixing Overfitting Issue và Improving Generalization

**Date:** 2026-03-04  
**Purpose:** Retrain model với proper hyperparameters để có thể detect synthetic attack patterns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn import tree
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    precision_recall_fscore_support
)
import joblib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# Tải dữ liệu
print("="*70)
print("LOADING DATASET")
print("="*70)

df_full = pd.read_csv('dataset5g-nidd\\Encoded\\Encoded.csv')
print(f"Total rows: {len(df_full):,}")
print(f"Total columns: {len(df_full.columns)}")
print(f"\nFirst few column names: {df_full.columns.tolist()[:10]}")

In [ ]:
# Chọn features (24 features + Label)
columns_to_keep = [
    'Seq', 'Mean', 'sTos', 'sTtl', 'dTtl', 'sHops', 'TotBytes', 
    'SrcBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'SrcWin', 
    'TcpRtt', 'AckDat', 'Label', ' e        ', ' e d      ', 
    'icmp', 'tcp', 'CON', 'FIN', 'INT', 'REQ', 'RST', 'Status'
]

df = pd.read_csv('dataset5g-nidd\\Encoded\\Encoded.csv', usecols=columns_to_keep)
print(f"\n✓ Selected {len(columns_to_keep)} columns")
print(f"Features to keep: {columns_to_keep}")

# Move Label to last column
column_to_move = df.pop('Label')
df.insert(24, "Label", column_to_move)

print(f"\n✓ Label moved to last column")
print(df.head())

In [ ]:
# 🔍 PHÂN TÍCH DATA IMBALANCE
print("\n" + "="*70)
print("DATA IMBALANCE ANALYSIS")
print("="*70)

label_counts = df['Label'].value_counts()
print(f"\nClass distribution:")
for label, count in label_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {label:12s}: {count:8,} ({percentage:5.2f}%)")

# Calculate imbalance ratio
benign_count = label_counts.get('Benign', 0)
malicious_count = label_counts.get('Malicious', 0)
imbalance_ratio = benign_count / malicious_count if malicious_count > 0 else 0

print(f"\n⚠️ Imbalance ratio: {imbalance_ratio:.2f}:1 (Benign:Malicious)")
if imbalance_ratio > 2:
    print("   → Significant imbalance detected! Will use class_weight='balanced'")

# Visualize
plt.figure(figsize=(10, 5))
colors = ['#54A0FF', '#FF6B6B']
bars = plt.bar(label_counts.index, label_counts.values, color=colors, edgecolor='black', linewidth=1.5)
plt.xlabel('Class', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=12, fontweight='bold')
plt.title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
plt.xticks(fontsize=11)

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}\n({height/len(df)*100:.1f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Chuẩn bị dữ liệu
print("\n" + "="*70)
print("PREPARING DATA")
print("="*70)

# Separate features and labels
X = df.iloc[:, 0:24].values
y = df.iloc[:, 24].values

print(f"\nFeature matrix shape: {X.shape}")
print(f"Label vector shape: {y.shape}")
print(f"Features (24): {df.columns[0:24].tolist()}")

# Split: 60% train, 20% validation, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"\n✓ Data split:")
print(f"  Training set:   {len(X_train):6,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation set: {len(X_val):6,} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test set:       {len(X_test):6,} samples ({len(X_test)/len(X)*100:.1f}%)")

In [ ]:
# Standardize features
print("\n" + "="*70)
print("FEATURE SCALING")
print("="*70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Data standardized using StandardScaler")
print(f"Feature means (first 8): {scaler.mean_[:8]}")
print(f"Feature stds (first 8): {scaler.scale_[:8]}")

In [ ]:
# 🎯 HYPERPARAMETER TUNING
print("\n" + "="*70)
print("HYPERPARAMETER TUNING WITH GRID SEARCH")
print("="*70)

# Define parameter grid
param_grid = {
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [50, 100, 200],
    'min_samples_leaf': [20, 50, 100],
    'class_weight': ['balanced', None]
}

print(f"\nParameter grid:")
for param, values in param_grid.items():
    print(f"  {param:20s}: {values}")

print(f"\nTotal combinations: {np.prod([len(v) for v in param_grid.values()])}")
print(f"Running GridSearchCV with 3-fold cross-validation...")
print("This may take a few minutes...\n")

# Grid search
grid_search = GridSearchCV(
    tree.DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

import time
start = time.time()
grid_search.fit(X_train_scaled, y_train)
elapsed = time.time() - start

print(f"\n✓ Grid search completed in {elapsed:.2f} seconds")
print(f"\nBest parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest cross-validation score: {grid_search.best_score_:.4f}")

In [ ]:
# Train với best parameters
print("\n" + "="*70)
print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
print("="*70)

best_model = grid_search.best_estimator_

# Evaluate on validation set
y_val_pred = best_model.predict(X_val_scaled)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"\nValidation set accuracy: {val_accuracy:.4f}")
print(f"\nValidation set classification report:")
print(classification_report(y_val, y_val_pred, digits=4))

# Model info
print(f"\n" + "="*70)
print("MODEL INFORMATION")
print("="*70)
print(f"Max depth: {best_model.tree_.max_depth}")
print(f"Number of leaves: {best_model.tree_.n_leaves}")
print(f"Number of nodes: {best_model.tree_.node_count}")
print(f"Number of features: {best_model.n_features_in_}")

if best_model.tree_.max_depth < 50:
    print(f"\n✓ Good! Max depth = {best_model.tree_.max_depth} (prevents overfitting)")
else:
    print(f"\n⚠️ Warning: Max depth = {best_model.tree_.max_depth} might be too deep")

In [ ]:
# 📊 EVALUATE ON TEST SET
print("\n" + "="*70)
print("FINAL EVALUATION ON TEST SET")
print("="*70)

y_test_pred = best_model.predict(X_test_scaled)
y_test_proba = best_model.predict_proba(X_test_scaled)

test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"\nTest set accuracy: {test_accuracy:.4f}")

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

print(f"\nClassification Report:")
print(classification_report(y_test, y_test_pred, digits=4))

# Calculate detailed metrics
tn, fp, fn, tp = cm.ravel()
print(f"\nDetailed Confusion Matrix:")
print(f"  True Negatives:  {tn:8,}")
print(f"  False Positives: {fp:8,}")
print(f"  False Negatives: {fn:8,}")
print(f"  True Positives:  {tp:8,}")

In [ ]:
# 🧪 TEST WITH SYNTHETIC ATTACK PATTERNS
print("\n" + "="*70)
print("TESTING WITH SYNTHETIC ATTACK PATTERNS")
print("="*70)

# Feature names
feature_names = df.columns[0:24].tolist()

# Define synthetic test cases
synthetic_tests = [
    {
        'name': '🔴 ATTACK 1: DDoS Flood (High Seq + Low TTL)',
        'features': [999999, 5000, 0, 1, 1, 50, 5000000, 2500000, 100, 1500, 1500, 65535, 500, 500000, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1],
        'expected': 'Malicious'
    },
    {
        'name': '🔴 ATTACK 2: Spoofing (Extreme TTL + ICMP)',
        'features': [1, 5000, 255, 255, 255, 100, 5000000, 2500000, 100, 1500, 1500, 65535, 500, 500000, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1],
        'expected': 'Malicious'
    },
    {
        'name': '🔴 ATTACK 3: High Traffic (Large bytes + RST)',
        'features': [80000, 3500, 8, 35, 35, 25, 1500000, 750000, 50, 1450, 1450, 65535, 350, 300000, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1],
        'expected': 'Malicious'
    },
    {
        'name': '🟢 NORMAL 1: Typical IoT Traffic',
        'features': [40000, 400, 0, 64, 64, 5, 3600, 2500, 50, 400, 400, 16000, 30, 5000, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1],
        'expected': 'Benign'
    },
    {
        'name': '🟢 NORMAL 2: Low Traffic',
        'features': [5000, 300, 0, 128, 128, 3, 15000, 8000, 50, 350, 350, 20000, 15, 3000, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1],
        'expected': 'Benign'
    },
]

print(f"\nTesting {len(synthetic_tests)} synthetic patterns...\n")

correct_predictions = 0
for test in synthetic_tests:
    features_scaled = scaler.transform(np.array(test['features']).reshape(1, -1))
    prediction = best_model.predict(features_scaled)[0]
    proba = best_model.predict_proba(features_scaled)[0]
    
    benign_prob = proba[0] if best_model.classes_[0] == 'Benign' else proba[1]
    malicious_prob = proba[1] if best_model.classes_[1] == 'Malicious' else proba[0]
    
    is_correct = (prediction == test['expected'])
    correct_predictions += int(is_correct)
    
    status = "✅ CORRECT" if is_correct else "❌ WRONG"
    
    print(f"{test['name']}")
    print(f"  Expected:   {test['expected']}")
    print(f"  Predicted:  {prediction} (Benign={benign_prob:.3f}, Malicious={malicious_prob:.3f})")
    print(f"  Status:     {status}")
    print()

accuracy_synthetic = (correct_predictions / len(synthetic_tests)) * 100
print(f"="*70)
print(f"✨ Synthetic Test Accuracy: {correct_predictions}/{len(synthetic_tests)} ({accuracy_synthetic:.1f}%)")
print(f"="*70)

if accuracy_synthetic >= 80:
    print("✓ EXCELLENT! Model can generalize to synthetic patterns!")
elif accuracy_synthetic >= 60:
    print("⚠️ MODERATE. Model partially works with synthetic data.")
else:
    print("❌ POOR. Model fails to generalize to synthetic patterns.")

In [ ]:
# 📊 VISUALIZE FEATURE IMPORTANCE
print("\n" + "="*70)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*70)

importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

print(f"\nTop 10 Most Important Features:")
for i in range(min(10, len(indices))):
    idx = indices[i]
    print(f"{i+1:2d}. {feature_names[idx]:20s}: {importances[idx]:.4f} ({importances[idx]*100:.2f}%)")

# Plot
plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, min(15, len(indices))))
top_15_indices = indices[:15]

plt.barh(range(len(top_15_indices)), importances[top_15_indices], color=colors, edgecolor='black', linewidth=0.8)
plt.yticks(range(len(top_15_indices)), [feature_names[i] for i in top_15_indices], fontsize=10)
plt.xlabel('Feature Importance', fontsize=12, fontweight='bold')
plt.title('Top 15 Feature Importance - Improved Decision Tree', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Check concentration
top_2_importance = importances[indices[0]] + importances[indices[1]]
if top_2_importance > 0.9:
    print(f"\n⚠️ Warning: Top 2 features account for {top_2_importance*100:.1f}% importance")
    print("   Model relies heavily on few features - consider using ensemble methods")
else:
    print(f"\n✓ Good distribution: Top 2 features = {top_2_importance*100:.1f}%")

In [ ]:
# Confusion Matrix Visualization
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'Malicious'], 
            yticklabels=['Benign', 'Malicious'],
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Improved Decision Tree', fontsize=14, fontweight='bold')
plt.ylabel('Actual Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba[:, 1], pos_label='Malicious')
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#2E86DE', linewidth=2, label=f'Decision Tree (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='#E74C3C', linestyle='--', linewidth=2, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.2, color='#2E86DE')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Improved Decision Tree', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"ROC AUC Score: {roc_auc:.4f}")

In [ ]:
# 💾 SAVE IMPROVED MODEL
print("\n" + "="*70)
print("SAVING IMPROVED MODEL")
print("="*70)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save to main model directory
import os
os.makedirs('../model', exist_ok=True)

model_path = f'../model/decision_tree_model_IMPROVED_{timestamp}.pkl'
scaler_path = f'../model/scaler_IMPROVED_{timestamp}.pkl'
features_path = f'../model/feature_names_IMPROVED_{timestamp}.pkl'

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(feature_names, features_path)

print(f"\n✓ Model saved to: {model_path}")
print(f"✓ Scaler saved to: {scaler_path}")
print(f"✓ Features saved to: {features_path}")

print(f"\n" + "="*70)
print("MODEL SUMMARY")
print("="*70)
print(f"Model Type:              Decision Tree Classifier (IMPROVED)")
print(f"Number of Features:      {len(feature_names)}")
print(f"Max Depth:               {best_model.tree_.max_depth}")
print(f"Min Samples Split:       {best_model.min_samples_split}")
print(f"Min Samples Leaf:        {best_model.min_samples_leaf}")
print(f"Class Weight:            {best_model.class_weight}")
print(f"\nPerformance:")
print(f"  Training Accuracy:     {grid_search.best_score_:.4f} (CV)")
print(f"  Validation Accuracy:   {val_accuracy:.4f}")
print(f"  Test Accuracy:         {test_accuracy:.4f}")
print(f"  ROC AUC:               {roc_auc:.4f}")
print(f"  Synthetic Test:        {accuracy_synthetic:.1f}%")
print(f"\n" + "="*70)

if accuracy_synthetic >= 80:
    print("✅ MODEL READY FOR PRODUCTION!")
    print("   This model can detect both real and synthetic attack patterns.")
else:
    print("⚠️ MODEL NEEDS FURTHER TUNING")
    print("   Consider ensemble methods or more training data.")

In [ ]:
# 📖 USAGE EXAMPLE
print("\n" + "="*70)
print("USAGE EXAMPLE")
print("="*70)

usage_code = f"""
# Load model in production
import joblib
import numpy as np

# Load files
model = joblib.load('{model_path}')
scaler = joblib.load('{scaler_path}')
features = joblib.load('{features_path}')

# Example: Predict new data
new_data = [
    80000,  # Seq
    3500,   # Mean
    8,      # sTos
    35,     # sTtl (LOW - suspicious!)
    35,     # dTtl
    25,     # sHops
    1500000,# TotBytes (HUGE - attack!)
    750000, # SrcBytes
    50,     # Offset
    1450,   # sMeanPktSz
    1450,   # dMeanPktSz
    65535,  # SrcWin
    350,    # TcpRtt (high latency)
    300000, # AckDat
    0, 1, 0, 1, 1, 0, 1, 1, 1, 1  # Flags (RST=1)
]

# Scale and predict
scaled_data = scaler.transform(np.array(new_data).reshape(1, -1))
prediction = model.predict(scaled_data)[0]
probability = model.predict_proba(scaled_data)[0]

print(f"Prediction: {{prediction}}")
print(f"Probability: Benign={{probability[0]:.3f}}, Malicious={{probability[1]:.3f}}")
"""

print(usage_code)

print("\n✨ Model training completed successfully!")
print("   Use the code above to integrate with your Edge Server.")